In [1]:
import sys
print(sys.executable)

import pandas as pd
import numpy as np
from pathlib import Path

print("All imports successful")

/home/piets/sirna_shrna_offtargets/.venv/bin/python
All imports successful


In [2]:
# Open GSE114273 associated RNA seq data

# Raw data
raw_path = Path(
    '../data/raw/GSE114273 (siPARP1_siPARP2)/GSE114273_raw_counts_GRCh38.p13_NCBI.tsv.gz')

# Normalized data
norm_path = Path(
    "../data/raw/GSE114273 (siPARP1_siPARP2)/GSE114273_norm_counts_TPM_GRCh38.p13_NCBI.tsv.gz")

In [3]:
# Load complete expression matrices
raw_counts = pd.read_csv(raw_path, sep="\t")
norm_counts = pd.read_csv(norm_path, sep="\t")

# Select samples from the siRNA experiment
selected_samples = [
    f"GSM{i}" for i in range(3138707, 3138717)
]

raw_counts = raw_counts[["GeneID"] + selected_samples].copy()
norm_counts = norm_counts[["GeneID"] + selected_samples].copy()

In [4]:
# Check expression matrices

print("Raw counts:", raw_counts.shape)
print("TPM:", norm_counts.shape)

print("Missing values (raw):", raw_counts.isna().sum().sum())
print("Missing values (TPM):", norm_counts.isna().sum().sum())

print("Identical GeneIDs:",
      raw_counts["GeneID"].equals(norm_counts["GeneID"]))

print("Identical sample columns:",
      raw_counts.columns.equals(norm_counts.columns))

display(raw_counts.head())

Raw counts: (39376, 11)
TPM: (39376, 11)
Missing values (raw): 0
Missing values (TPM): 0
Identical GeneIDs: True
Identical sample columns: True


,GeneID,GSM3138707,GSM3138708,GSM3138709,GSM3138710,GSM3138711,GSM3138712,GSM3138713,GSM3138714,GSM3138715,GSM3138716
0,100287102,1,1,1,1,1,0,1,1,1,1
1,653635,79,91,97,77,86,64,79,77,98,101
2,102466751,2,1,1,2,2,2,2,1,3,2
3,107985730,1,0,0,0,0,0,0,0,0,0
4,100302278,0,0,0,0,0,0,0,0,0,0


In [5]:
# Load and inspect annotation data

annotation_path = Path(
    "../data/raw/GSE114273 (siPARP1_siPARP2)/Human.GRCh38.p13.annot.tsv"
)

gene_annotation = pd.read_csv(
    annotation_path,
    sep="\t"
)

print("Dimensions:", gene_annotation.shape)
print("Columns:", gene_annotation.columns.tolist())

display(gene_annotation.head())

print(gene_annotation["GeneType"].value_counts())

Dimensions: (39376, 18)
Columns: ['GeneID', 'Symbol', 'Description', 'Synonyms', 'GeneType', 'EnsemblGeneID', 'Status', 'ChrAcc', 'ChrStart', 'ChrStop', 'Orientation', 'Length', 'GOFunctionID', 'GOProcessID', 'GOComponentID', 'GOFunction', 'GOProcess', 'GOComponent']


/tmp/ipykernel_1468/1407269604.py:7: DtypeWarning: Columns (8: ChrStart, 9: ChrStop) have mixed types. Specify dtype option on import or set low_memory=False.
  gene_annotation = pd.read_csv(


,GeneID,Symbol,Description,Synonyms,GeneType,EnsemblGeneID,Status,ChrAcc,ChrStart,ChrStop,Orientation,Length,GOFunctionID,GOProcessID,GOComponentID,GOFunction,GOProcess,GOComponent
0,100287102,DDX11L1,DEAD/H-box helicase 11 like 1 (pseudogene),NaN,pseudo,ENSG00000290825,active,NC_000001.11,11874,14409,positive,1652,NaN,NaN,NaN,NaN,NaN,NaN
1,653635,WASH7P,"WASP family homolog 7, pseudogene",FAM39F|WASH5P,pseudo,NaN,active,NC_000001.11,14362,29370,negative,1769,NaN,NaN,NaN,NaN,NaN,NaN
2,102466751,MIR6859-1,microRNA 6859-1,hsa-mir-6859-1,ncRNA,ENSG00000278267,active,NC_000001.11,17369,17436,negative,68,NaN,NaN,NaN,NaN,NaN,NaN
3,107985730,MIR1302-2HG,MIR1302-2 host gene,NaN,ncRNA,NaN,active,NC_000001.11,29926,31295,positive,538,NaN,NaN,NaN,NaN,NaN,NaN
4,100302278,MIR1302-2,microRNA 1302-2,MIRN1302-2|hsa-mir-1302-2,ncRNA,ENSG00000284332,active,NC_000001.11,30366,30503,positive,138,NaN,GO:0035195,NaN,NaN,miRNA-mediated gene silencing,NaN


GeneType
protein-coding    19416
ncRNA             15480
pseudo             1392
snoRNA              541
tRNA                437
other               388
snRNA                63
rRNA                 39
unknown               6
scRNA                 4
Name: count, dtype: int64


In [6]:
print(gene_annotation[
    ["GeneID", "EnsemblGeneID", "Symbol", "GeneType", "Description"]
].info())

<class 'pandas.DataFrame'>
RangeIndex: 39376 entries, 0 to 39375
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   GeneID         39376 non-null  int64
 1   EnsemblGeneID  27077 non-null  str  
 2   Symbol         39376 non-null  str  
 3   GeneType       37766 non-null  str  
 4   Description    38059 non-null  str  
dtypes: int64(1), str(4)
memory usage: 1.5 MB
None


In [11]:
# Extract relevant annotation columns
gene_mapping = gene_annotation[
    ["GeneID", "EnsemblGeneID", "Symbol", "GeneType", "Description"]
].copy()

# Retain protein-coding genes for analysis
gene_mapping = gene_mapping[
    gene_mapping["GeneType"] == "protein-coding"
].copy()

# Remove entries without Ensembl IDs
gene_mapping = gene_mapping.dropna(
    subset=["EnsemblGeneID"]
)

print("Protein-coding genes:", len(gene_mapping))
print("Unique Entrez IDs:", gene_mapping["GeneID"].nunique())
print("Unique Ensembl IDs:", gene_mapping["EnsemblGeneID"].nunique())

display(gene_mapping.head())

Protein-coding genes: 19238
Unique Entrez IDs: 19238
Unique Ensembl IDs: 19227


,GeneID,EnsemblGeneID,Symbol,GeneType,Description
6,79501,ENSG00000186092,OR4F5,protein-coding,olfactory receptor family 4 subfamily F member 5
14,729759,ENSG00000284733,OR4F29,protein-coding,olfactory receptor family 4 subfamily F member 29
19,81399,ENSG00000284662,OR4F16,protein-coding,olfactory receptor family 4 subfamily F member 16
34,148398,ENSG00000187634,SAMD11,protein-coding,sterile alpha motif domain containing 11
35,26155,ENSG00000188976,NOC2L,protein-coding,NOC2 like nucleolar associated transcriptional...


In [12]:
# Entrez IDs mapping to duplicated Ensembl IDs
duplicates = gene_mapping[
    gene_mapping["EnsemblGeneID"].duplicated(keep=False)
].sort_values("EnsemblGeneID")

display(duplicates)

,GeneID,EnsemblGeneID,Symbol,GeneType,Description
4437,51130,ENSG00000115239,ASB3,protein-coding,ankyrin repeat and SOCS box containing 3
4438,100302652,ENSG00000115239,GPR75-ASB3,protein-coding,GPR75-ASB3 readthrough
28033,56924,ENSG00000137843,PAK6,protein-coding,p21 (RAC1) activated kinase 6
28032,106821730,ENSG00000137843,BUB1B-PAK6,protein-coding,BUB1B-PAK6 readthrough
12424,51256,ENSG00000145979,TBC1D7,protein-coding,TBC1 domain family member 7
12422,107080638,ENSG00000145979,TBC1D7-LOC100130357,protein-coding,TBC1D7-LOC100130357 readthrough
15140,109504726,ENSG00000152926,ERV3-1-ZNF117,protein-coding,ERV3-1-ZNF117 readthrough
15141,51351,ENSG00000152926,ZNF117,protein-coding,zinc finger protein 117
471,100532736,ENSG00000158747,MICOS10-NBL1,protein-coding,MICOS10-NBL1 readthrough
473,4681,ENSG00000158747,NBL1,protein-coding,"NBL1, DAN family BMP antagonist"


In [13]:
# Identify readthrough entries among duplicated Ensembl IDs
duplicate_ensembl_ids = gene_mapping.loc[
    gene_mapping["EnsemblGeneID"].duplicated(keep=False),
    "EnsemblGeneID"
].unique()

readthrough_duplicates = gene_mapping[
    (gene_mapping["EnsemblGeneID"].isin(duplicate_ensembl_ids))
    & gene_mapping["Description"].str.contains(
        "readthrough", case=False, na=False
    )
]

# Identify the additional entry
additional_exclusion = gene_mapping[
    gene_mapping["GeneID"] == 102723728
]

# Combine entries to remove
removed_duplicates = pd.concat(
    [readthrough_duplicates, additional_exclusion]
).drop_duplicates(subset="GeneID")

# Remove them from the mapping
gene_mapping = gene_mapping[
    ~gene_mapping["GeneID"].isin(removed_duplicates["GeneID"])
].copy()

In [14]:
print("Removed entries:", len(removed_duplicates))
print("Final gene mapping:", len(gene_mapping))
print("Duplicate Ensembl IDs:", gene_mapping["EnsemblGeneID"].duplicated().sum())

Removed entries: 11
Final gene mapping: 19227
Duplicate Ensembl IDs: 0


#### Gene annotation filtering

The NCBI gene annotation table was filtered to retain protein-coding genes only. Entries without an Ensembl Gene ID (178 entries) were excluded to ensure consistent mapping between gene identifiers and transcript annotations.

Duplicate Ensembl Gene IDs were then investigated. Ten readthrough entries involved in duplicated mappings were removed, along with one additional entry (GeneID: 102723728) identified during manual inspection. Other readthrough entries with unique Ensembl mappings were retained.

This resulted in a final mapping of **19,227 protein-coding genes**, with unique Ensembl Gene IDs.

In [15]:
processed_dir = Path("../data/processed/GSE114273 (siPARP1_siPARP2)")
processed_dir.mkdir(parents=True, exist_ok=True)

In [20]:
assert raw_counts.shape[1] == 11  # GeneID + 10 samples
assert norm_counts.shape[1] == 11
assert raw_counts["GeneID"].equals(norm_counts["GeneID"])

assert len(gene_mapping) == 19227
assert gene_mapping["EnsemblGeneID"].notna().all()
assert not gene_mapping["EnsemblGeneID"].duplicated().any()

print("All checks passed.")
print(gene_mapping.shape)

All checks passed.
(19227, 5)


In [22]:
# save cleaned gene_mapping dataframe

gene_mapping.to_csv(
    processed_dir / "GSE114273_gene_mapping_protein_coding.tsv",
    sep="\t", index=False
)